# Unit 9: TensorBoard 实战 — CIFAR-10 全流程日志

## 学习目标
- 在真实 CIFAR-10 训练中全面应用 TensorBoard
- 记录 Scalars、Histograms、Graph、Images、Text、HParams
- 掌握训练循环中的日志插入策略
- 学会错误分析和特征图可视化方法
- 在 Jupyter 中启动 TensorBoard 查看结果


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 9.1 实战：完整的 CIFAR-10 训练 + TensorBoard 全方位日志

我们将构建一个小型 CNN 并在训练过程中把**所有可观测信息**都记录到 TensorBoard。

### 记录清单
- **Scalars**: 每个 epoch 的 train/val loss 和 accuracy，学习率
- **Scalars (per batch)**: 每个 batch 的训练损失（细粒度曲线）
- **Histograms**: 每层的权重和梯度分布
- **Graph**: 模型计算图
- **Images**: 训练样本（含数据增强效果）、特征图、错误分类样本
- **Text**: 每个 epoch 的文本摘要
- **HParams**: 所有超参数 + 最终指标

In [ ]:
exp_name = f"cifar10_cnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
log_dir_exp = Path("runs") / exp_name
log_dir_exp.mkdir(parents=True, exist_ok=True)
print(f"Experiment log: {log_dir_exp}")

tb_writer = SummaryWriter(str(log_dir_exp))

### 9.1.1 数据准备

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

full_train = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=test_transform)

train_size = 45000
val_size = 5000
train_set, val_set = random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f"Train: {train_size:,} | Val: {val_size:,} | Test: {len(test_dataset):,}")

### 9.1.2 记录数据增强效果

这是一个简洁实用的CIFAR-10图像逆标准化工具函数，核心思想就是对每个通道执行 x * std + mean 并裁剪到 [0,1]。

In [ ]:
def denorm(img, mean=cifar10_mean, std=cifar10_std):
    img = img.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return img.clamp_(0, 1)

raw_train = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
aug_samples = torch.stack([denorm(raw_train[i][0]) for i in range(16)])
tb_writer.add_images("Data/Augmented_Samples", aug_samples, 0)
print("Augmented samples recorded to TensorBoard.")

### 9.1.3 模型定义 + 记录计算图

In [ ]:
class TB_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = TB_CNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

dummy_graph = torch.randn(1, 3, 32, 32).to(device)
tb_writer.add_graph(model, dummy_graph)
print("Model graph recorded.  ✔")

In [ ]:
# 只打印有参数的模块
for name,param in model.named_parameters():
    print(name)
    print("="*50)


### 9.1.4 带 TensorBoard 日志的训练循环

这是本单元的核心——展示如何在实际训练中全面使用 TensorBoard。

注意 `writer.add_scalar` 和 `writer.add_histogram` 的 `global_step` 参数：
- 它确定数据在时间轴上的位置
- 可以使用 epoch 数或全局 batch 计数

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

def train_one_epoch(model, loader, optimizer, criterion, epoch, writer, global_batch):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)

        writer.add_scalar("Loss/train_batch", loss.item(), global_batch)
        writer.add_scalar("Accuracy/train_batch", pred.eq(target).float().mean().item(), global_batch)
        global_batch += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total, correct / total, global_batch

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
        all_preds.append(pred.cpu())
        all_labels.append(target.cpu())
    return total_loss / total, correct / total, torch.cat(all_preds), torch.cat(all_labels)

EPOCHS = 25
global_batch = 0
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc, global_batch = train_one_epoch(
        model, train_loader, optimizer, criterion, epoch, tb_writer, global_batch,
    )
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion)
    scheduler.step() # per-epoch

    tb_writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    tb_writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, epoch)
    tb_writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    for name, param in model.named_parameters():
        tb_writer.add_histogram(f"Weights/{name}", param.data, epoch)
        if param.grad is not None:
            tb_writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

    tb_writer.add_text(
        "Training_Summary",
        f"""| Epoch | Train Loss | Train Acc | Val Loss | Val Acc |
|:-----:|:----------:|:---------:|:--------:|:-------:|
| {epoch+1:5d} |   {train_loss:.4f}  |  {train_acc:.4f}  | {val_loss:.4f} | {val_acc:.4f} |""",
        epoch,
    )

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), str(log_dir_exp / "best_model.pt"))
        print(f"  ➤ Best model saved! (Val Acc: {val_acc:.4f})")

print(f"\nTraining complete. Best Val Acc: {best_val_acc:.4f}")

### 9.1.5 记录错误分类样本

把验证集上预测错误的样本记录下来，方便分析模型弱点。

In [ ]:
model.load_state_dict(torch.load(log_dir_exp / "best_model.pt", map_location=device, weights_only=False))
model.eval()

# 仅张量化，不归一化，不裁切
raw_val = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())


"""val_set 是什么？原始 dataset 中的真实下标
它是 random_split(full_train, [45000, 5000]) 返回的第二个对象，类型为 torch.utils.data.Subset。它本身不存储任何图片数据，只存储了两样东西：
对原始数据集 full_train 的引用
一个整数索引列表（即 .indices）
.indices 返回什么？
返回一个长度为 5000 的列表/张量，每个元素是 full_train 中的下标（范围 0~49999）。
"""
val_indices = val_set.indices

mis_img = []   # 存放误分类样本的原始图片（未归一化）
mis_true = []  # 存放真实标签字符串，如 "True:cat"
mis_pred = []  # 存放预测标签字符串，如 "Pred:dog"
for i, (data, target) in enumerate(val_loader):
    data, target = data.to(device), target.to(device)
    output = model(data)
    pred = output.argmax(dim=1)
    wrong_mask = pred != target
    if wrong_mask.any():
        mis_local_indices = wrong_mask.nonzero(as_tuple=True)[0]  # ← 只算一次
        for j in range(mis_local_indices.shape[0]):
            if len(mis_img) >= 16:
                break
            idx = (i * 128 + mis_local_indices[j].item()) % len(val_indices)
            orig_idx = val_indices[idx]
            orig_img, _ = raw_val[orig_idx]
            mis_img.append(orig_img)
            mis_true.append(f"True:{raw_val.classes[target[mis_local_indices[j]].item()]}")
            mis_pred.append(f"Pred:{raw_val.classes[pred[mis_local_indices[j]].item()]}")
        if len(mis_img) >= 16:
            break

if mis_img:
    grid = torch.stack(mis_img)
    tb_writer.add_images("Error_Analysis/Misclassified", grid, 0)
    for k, (t, p) in enumerate(zip(mis_true, mis_pred)):
        tb_writer.add_text("Error_Analysis/Labels", f"{k:2d}: {t} | {p}", k)
    print(f"Recorded {len(mis_img)} misclassified samples.")

### 9.1.6 记录特征图可视化

In [ ]:
@torch.no_grad()
def log_feature_maps(model, input_tensor, writer, tag_prefix, step):
    model.eval()
    features = {}
    def hook_fn(name):
        def hook(module, inp, out):
            features[name] = out.detach()
        return hook

    hooks = []
    for name, module in model.features.named_children():
        if isinstance(module, nn.ReLU) or isinstance(module, nn.MaxPool2d):
            continue
        hooks.append(module.register_forward_hook(hook_fn(name)))

    _ = model(input_tensor)
    for h in hooks:
        h.remove()

    for name, fmap in features.items():
        fmap = fmap[0].unsqueeze(1)
        n_show = min(16, fmap.size(0))
        fmap_show = fmap[:n_show].repeat(1, 3, 1, 1)
        fmap_show = (fmap_show - fmap_show.min()) / (fmap_show.max() - fmap_show.min() + 1e-8)
        writer.add_images(f"{tag_prefix}/{name}", fmap_show, step)

sample_input = next(iter(train_loader))[0][0:1].to(device)  # 从数据集中取第一张图片
# 保存这张照片原图，不归一化
tb_writer.add_image("Error_Analysis/Sample", denorm(sample_input[0]), 0)

log_feature_maps(model, sample_input, tb_writer, "Feature_Maps", 5)
print("Feature maps recorded to TensorBoard.")

### 9.1.7 记录 HParams + 最终测试结果

In [ ]:
test_loss, test_acc, _, _ = validate(model, test_loader, criterion)
print(f"Test Acc: {test_acc:.4f} ({test_acc*100:.2f}%)")

hparams_dict = {
    "lr": 0.05,
    "batch_size": 128,
    "optimizer": "SGD",
    "scheduler": "CosineAnnealingLR",
    "weight_decay": 5e-4,
    "epochs": EPOCHS,
    "model_type": "TB_CNN",
}

metrics_dict = {
    "hparam/best_val_acc": best_val_acc,
    "hparam/test_acc": test_acc,
    "hparam/test_loss": test_loss,
}

tb_writer.add_hparams(hparams_dict, metrics_dict)
print("HParams recorded.")

## 9.2 关闭 Writer

完成所有日志记录后，记得关闭 `SummaryWriter`。或者使用 `with` 语句自动管理。

In [ ]:
tb_writer.close()
print("TensorBoard writer closed.")

## 9.3 启动 TensorBoard 查看结果

运行下面 cell 在 Jupyter 中直接查看 TensorBoard，或运行上面的终端命令。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs --port 6006
# %reload_ext tensorboard

## 9.4 进阶技巧

### 技巧 1：为多个实验建立命名规范
```python
log_dir = f"runs/{model_name}_lr{lr}_bs{batch_size}_{timestamp}"
```
这样在 TensorBoard 的 `runs` 选择器中可以按中文/英文名快速区分。

### 技巧 2：用 `add_scalars` 做同图对比
- 训练损失 / 验证损失 放同一张图
- 不同层的梯度范数 放同一张图

### 技巧 3：定期记录而不是每 batch
每 batch 记录一次会让日志文件巨大。建议：
- **每 batch** 记录 `Loss/train_batch`（可选，需要细粒度调试时开）
- **每 epoch** 记录 histogram、images
- **每 N 个 epoch** 记录特征图和 embeddings

### 技巧 4：TensorBoard.dev 云端分享
```bash
tensorboard dev upload --logdir runs \
    --name "CIFAR-10 CNN Experiment" \
    --description "Comparing different architectures"
```
可以生成一个公开链接，方便与团队分享实验结果。

### 技巧 5：日志清理
TensorBoard 日志文件可能变得很大。定期清理：
```python
import shutil
shutil.rmtree("runs/old_experiment")  # 删除不需要的实验
```

## 9.5 单元小结

| 功能 | API | 最佳实践 |
|------|-----|---------|
| **Scalars** | `add_scalar` / `add_scalars` | 用 `/` 分层命名，train/val 同图 |
| **Histograms** | `add_histogram` | 每 epoch 记录，观察分布演变 |
| **Graph** | `add_graph` | 训练前记录一次 |
| **Images** | `add_images` | 记录样本、增强效果、错误分类 |
| **HParams** | `add_hparams` | 训练结束时记录，配合多实验对比 |
| **Text** | `add_text` | 记录 epoch 摘要、实验配置 |

### TensorBoard vs 手动绘图

| 方面 | `matplotlib` | TensorBoard |
|------|:-----------:|:-----------:|
| 设置成本 | 低 | 中 |
| 实时刷新 | ❌ 需要重新绘图 | ✅ 自动刷新 |
| 多实验对比 | ❌ 手动叠加 | ✅ 自动分组 |
| 权重/梯度分布 | ❌ 不支持 | ✅ Histogram |
| 计算图 | ❌ 不支持 | ✅ Graph 面板 |
| 超参数搜索 | ❌ 不支持 | ✅ HParams 面板 |
| 远端分享 | ❌ 需导出图片 | ✅ tensorboard.dev |
| 论文图 | ✅ 高质量原生 | ❌ 需截图处理 |

> **建议**：训练时用 TensorBoard 做实时监控，最终结果用 matplotlib 导出高质量论文图。两者互补，不是二选一。

### 思考题
1. 为什么建议只在每 epoch 记录 histogram 而非每 batch？
2. `add_scalar` 和 `add_scalars` 的区别是什么？什么时候用哪个？
3. 如果训练了 100 组超参数，如何在 TensorBoard 中找到最优的一组？
4. TensorBoard 的日志文件本质是什么格式？可以在 Notebook 里直接读取吗？